# 0 — Prepare slim h5ads for D2/D4/D10 integration

Streams one dataset at a time, drops unused obsm/obsp/uns, ensures sparse counts, writes slim files to `INTEGRATION_DIR/slim/`. Designed to keep peak RSS low on a 16 GB desktop.

Output: `data/data-objects/integration/slim/<dataset>.h5ad` for each of D10_Lapa, D2_DZ, D2_Lapa, D4_DZ, D4_Lapa.

In [ ]:
import sys, gc, os
from pathlib import Path

_p = Path('.').resolve()
while not (_p / 'src' / 'config.py').exists() and _p != _p.parent:
    _p = _p.parent
sys.path.insert(0, str(_p))

from src.config import DATASETS, INTEGRATION_DIR
from src.transfer_learning import prepare_slim_h5ad

import scanpy as sc

try:
    import psutil
    proc = psutil.Process()
    def rss_gb():
        return proc.memory_info().rss / 1024**3
except ImportError:
    def rss_gb():
        return float('nan')

print('peak RSS at start: %.2f GB' % rss_gb())

In [ ]:
SLIM_DIR = INTEGRATION_DIR / 'slim'
SLIM_DIR.mkdir(parents=True, exist_ok=True)

# Reference: prefer the labelled CellAssign h5ad so we get `initial_cellassign_prediction`.
# Queries: clustered files (no labels yet).
DATASET_PLAN = [
    ('D10_Lapa', 'labelled'),
    ('D2_DZ',    'clustered'),
    ('D2_Lapa',  'clustered'),
    ('D4_DZ',    'clustered'),
    ('D4_Lapa',  'clustered'),
]
DATASET_PLAN

In [ ]:
# Stream: one dataset in RAM at a time
for key, stage in DATASET_PLAN:
    out = SLIM_DIR / f'{key}.h5ad'
    print(f'\n=== {key} (stage={stage}) ===')
    prepare_slim_h5ad(
        dataset_key=key,
        out_path=out,
        gene_list=None,        # Notebook 1 does the HVG-subset emit
        prefer_stage=stage,
    )
    gc.collect()
    print(f'RSS after {key}: %.2f GB' % rss_gb())

In [ ]:
# Sanity print: cells, sparse density, layer integrity
import numpy as np
from scipy import sparse

for key, _ in DATASET_PLAN:
    a = sc.read_h5ad(SLIM_DIR / f'{key}.h5ad', backed='r')
    counts = a.layers['counts']
    n_obs, n_vars = a.shape
    nnz = counts.nnz if hasattr(counts, 'nnz') else int((counts != 0).sum())
    density = nnz / (n_obs * n_vars)
    obs_cols = list(a.obs.columns)
    print(f'{key}: {n_obs} cells x {n_vars} genes, density={density:.3%}, obs={obs_cols[:8]}...')
    a.file.close()

In [ ]:
# Verify counts are integer-valued (a common failure mode is double-normalised exports)
for key, _ in DATASET_PLAN:
    a = sc.read_h5ad(SLIM_DIR / f'{key}.h5ad')
    sample = a.layers['counts'][:1000].toarray() if hasattr(a.layers['counts'], 'toarray') else a.layers['counts'][:1000]
    is_int = np.allclose(sample, sample.astype(int))
    print(f'{key}: counts integer? {is_int}; max={sample.max():.2f}')
    del a

Slim files written. Continue with `1_Reference_HVGs_and_Training.ipynb`.